# Lesson 65 · 🎨 Training visualized — an X-ray of the training process: loss curves, gradient norms and weight histograms

**Goal**: give the training process a health check — loss curves, gradient norms, weight histograms, a confusion matrix.

> **🎨**: less writing, more looking at shapes; the curves themselves are the answer.

🔗 **Aurora connection**:
- the `Value` computation graph → `L54_value_autograd`, `L55_forward_pass`
- the training loop → `L58_training_loop`
- feature extraction → the mel filterbank in `src/aurora/audio/mel.py`
- the model backbone → the KeywordCNN defined in L63


← **Previous lesson** · [L64 · Training and evaluation](L64_kws_train_eval.ipynb)

> Last lesson covered **training and evaluation**: train loop + validation loop, accuracy / confusion matrix / diagnosing overfitting.
> This lesson explores **training visualized**.

Visualization isn't decoration; it's the dashboard of the training process. The computation graph tells you where the gradient flows and where it breaks; the loss curve tells you whether the model is underfitting or has memorized the training set; the confusion matrix tells you which pair of keywords gets mixed up, pointing to a specific weakness in the feature layer. Put the three pictures together and you can usually pin down the root cause of a failed training run.

In [ ]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import itertools

## 1. The DAG structure of the Value computation graph

Every time you write `c = a + b`, a directed acyclic graph (DAG) is built behind the scenes:

```
a ──┐
    ├─[+]── c ──┐
b ──┘            ├─[*]── e
                 │
d ───────────────┘
```

- **nodes**: each `Value` object, storing `data` (the forward value) and `grad` (the backward gradient)
- **edges**: the operators (`+`, `*`, `**`), pointing in the direction of data flow
- **`_prev`**: each node records the set of its direct input nodes; backpropagation walks back along `_prev`

Drawing the DAG with matplotlib: draw nodes as rounded boxes and edges as arrows, write `data` and `grad` inside each box and the operator name outside it. No graphviz needed — pure matplotlib, so it runs anywhere.

In [ ]:
# ── Demo: draw the DAG of a Value computation graph with matplotlib ──
# A small graph: L = (a * b) + b**2, a=2, b=3
# L = 6 + 9 = 15; dL/da = b = 3; dL/db = a + 2b = 2+6 = 8

nodes = {
    'a':    {'data':  2.0, 'grad': 3.0, 'op': ''},
    'b':    {'data':  3.0, 'grad': 8.0, 'op': ''},
    'a*b':  {'data':  6.0, 'grad': 1.0, 'op': '*'},
    'b**2': {'data':  9.0, 'grad': 1.0, 'op': '**2'},
    'L':    {'data': 15.0, 'grad': 1.0, 'op': '+'},
}
edges = [
    ('a',    'a*b'),
    ('b',    'a*b'),
    ('b',    'b**2'),
    ('a*b',  'L'),
    ('b**2', 'L'),
]
pos = {
    'a':    (0, 2),
    'b':    (0, 0),
    'a*b':  (2, 2),
    'b**2': (2, 0),
    'L':    (4, 1),
}

fig, ax = plt.subplots(figsize=(9, 4))
ax.set_xlim(-0.8, 5.2)
ax.set_ylim(-0.8, 3.2)
ax.axis('off')
ax.set_title('Computation graph of L = (a * b) + b**2 (forward value | gradient)', fontsize=13)

BOX_W, BOX_H = 1.1, 0.7
leaf_color = '#d4e6f1'
op_color   = '#fdebd0'
for name, (x, y) in pos.items():
    is_leaf = nodes[name]['op'] == ''
    fc = leaf_color if is_leaf else op_color
    rect = mpatches.FancyBboxPatch(
        (x - BOX_W/2, y - BOX_H/2), BOX_W, BOX_H,
        boxstyle='round,pad=0.05', facecolor=fc, edgecolor='#555', linewidth=1.5
    )
    ax.add_patch(rect)
    d  = nodes[name]['data']
    g  = nodes[name]['grad']
    op = nodes[name]['op']
    label = f'{name}\ndata={d:.1f} | grad={g:.1f}'
    ax.text(x, y + 0.05, label, ha='center', va='center', fontsize=8.5)
    if op:
        ax.text(x, y - BOX_H/2 - 0.18, op, ha='center', va='top',
                fontsize=9, color='#c0392b', fontweight='bold')

for src, dst in edges:
    x0, y0 = pos[src]
    x1, y1 = pos[dst]
    dx, dy = x1 - x0, y1 - y0
    shrink = BOX_W / 2 + 0.05
    length = (dx**2 + dy**2) ** 0.5
    ux, uy = dx / length, dy / length
    ax.annotate('',
        xy=(x1 - ux * shrink, y1 - uy * shrink),
        xytext=(x0 + ux * shrink, y0 + uy * shrink),
        arrowprops=dict(arrowstyle='->', color='#2c3e50', lw=1.4)
    )

# Legend handles: a Patch here is only a stand-in and can't be add_patch-ed to the axes;
# passing it straight to legend as a handle is enough to draw the color swatch.
legend_handles = [
    mpatches.Patch(facecolor=leaf_color, edgecolor='#555', label='leaf node (input variable)'),
    mpatches.Patch(facecolor=op_color,  edgecolor='#555', label='operator node'),
]
ax.legend(handles=legend_handles, loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()
print('✅ DAG drawn; blue = leaf nodes, orange = operator nodes; grad holds the values backward() would fill in (worked out by hand here)')

### 🔍 Backpropagation step by step — taking the chain rule apart

The DAG above shows $\frac{\partial L}{\partial a} = 3$ and $\frac{\partial L}{\partial b} = 8$. Where do these numbers come from?

**The core principle**: backpropagation walks the DAG in reverse along `_prev`, accumulating gradients step by step with the **chain rule**.

#### Example: $L = (a \times b) + b^2$, with $a=2, b=3$

**Forward pass** (record data):
```
intermediate  c = a * b  = 2 * 3 = 6
intermediate  d = b**2   = 3**2 = 9
final         L = c + d  = 6 + 9 = 15
```

**Backward pass** (starting from $\frac{\partial L}{\partial L} = 1$):

**Step 1**: Loss → gradients of $c$ and $d$

$$\frac{\partial L}{\partial c} = \frac{\partial L}{\partial L} \times \frac{\partial L}{\partial c} = 1 \times 1 = 1 \quad \text{(because $L = c + d$, $\frac{\partial L}{\partial c} = 1$)}$$

$$\frac{\partial L}{\partial d} = \frac{\partial L}{\partial L} \times \frac{\partial L}{\partial d} = 1 \times 1 = 1 \quad \text{(addition has derivative 1 for each term)}$$

**Step 2**: send $\frac{\partial L}{\partial c}$ back to $a$ and $b$ (through the multiplication)

$$\frac{\partial L}{\partial a} = \frac{\partial L}{\partial c} \times \frac{\partial c}{\partial a} = 1 \times b = 1 \times 3 = 3 \quad \text{(chain rule for a product: $\frac{\partial (a \times b)}{\partial a} = b$)}$$

$$\frac{\partial L}{\partial b}\big|_{\text{via }c} = \frac{\partial L}{\partial c} \times \frac{\partial c}{\partial b} = 1 \times a = 1 \times 2 = 2$$

**Step 3**: send $\frac{\partial L}{\partial d}$ back to $b$ (through the square)

$$\frac{\partial L}{\partial b}\big|_{\text{via }d} = \frac{\partial L}{\partial d} \times \frac{\partial d}{\partial b} = 1 \times 2b = 1 \times 2 \times 3 = 6 \quad \text{(power rule: $\frac{\partial (b^2)}{\partial b} = 2b$)}$$

**Step 4**: $\frac{\partial L}{\partial b}$ comes from two paths — add them

$$\frac{\partial L}{\partial b} = \frac{\partial L}{\partial b}\big|_{\text{via }c} + \frac{\partial L}{\partial b}\big|_{\text{via }d} = 2 + 6 = 8$$

---

**Summary**:
- the gradient of a **leaf node** $a, b$ = the product of partial derivatives along each path through it, summed over all paths
- the gradient of an **operator node** (such as $c, d$) = the gradient flowing into it from the output
- `Value.backward()` in the code is exactly this chain rule, automated



## 2. Loss curves: spotting the early signs of overfitting

The two most important curves during training:

- `train_loss`: the model's mean squared error (or cross-entropy) on the training set; it **should fall steadily**
- `val_loss`: the same metric on the validation set; it **follows it down at first, then turns up once overfitting starts**

The difference between them is called the **generalization gap**:

```
gap(epoch) = val_loss(epoch) - train_loss(epoch)
```

**The signal for early stopping**: when `val_loss` hasn't fallen for `patience` epochs in a row, overfitting is taken to have begun: stop training and keep the weights from the best validation epoch.

For keyword spotting (KeywordCNN), common causes of overfitting:
- the training set has a single acoustic environment while the validation set has noise
- the model's capacity (number of convolution kernels) is far larger than the number of training samples
- the mel feature dimension (`n_mels=40`) is too high relative to the number of samples

### 🎢 How is the loss curve below "made up"? — taking every term of the formula apart

Before looking at the code below, picture something more familiar: **riding a bike downhill**. At the start the slope is steep; you barely pedal and your speed shoots up. The further down you go, the gentler the slope, and the **increase** in speed gets smaller and smaller; when you reach the flat, your speed stops changing. The training loss curve of a real neural network has much the same shape — at first the parameters are far from the optimum, so any gradient-descent step cuts a big chunk off the loss; the closer you get to the optimum, the less room there is to push it down, and progress gets slower and slower.

Mathematically, "the rate of decrease is proportional to what's left" is written as **exponential decay**: $y = y_0 \cdot e^{-kt}$ (the larger $k$, the faster the decay). That's why the line of code below looks the way it does:

```python
train_loss = 1.0 * np.exp(-0.08 * epochs) + 0.05 + 0.01 * np.random.randn(60)
```

Term by term:

- **`1.0 * np.exp(-0.08 * epochs)`** — this is the "ever slower descent" itself. `1.0` is the rough size of the initial loss (in the first epoch the model is basically guessing); `0.08` is a "decay speed" we made up, and the larger it is, the faster the curve drops. **It doesn't correspond to any real model's learning rate**; it's only there to make the curve over 60 epochs look like a typical training curve — change 0.08 to 0.2 and you'll see the curve flatten out within the first few epochs.
- **`+ 0.05`** — this is the "floor" the curve approaches but **never gets below**. In real training the loss can't reach zero: the data itself is noisy and some samples are genuinely ambiguous (e.g. speech with heavy background noise). This lower limit you can't push past has a name: **irreducible error**. The `0.05` here is just an illustrative value we picked.
- **`+ 0.01 * np.random.randn(60)`** — this is **the random jitter of training itself**. Real training doesn't look at all the data before each update; it draws a small batch (mini-batch) each time to estimate the gradient direction, and that estimate is naturally noisy. So even when the model is steadily improving, the loss curve isn't a smooth downward line but "downward overall, jittery locally". `np.random.randn(60)` generates 60 random jitters and `0.01` controls their size.
- **`np.random.seed(42)`** (at the very top of this code) — fixes the random seed, so every time you re-run it the jitter values are exactly the same, which makes comparing and debugging easier.

`val_loss` is built the same way ("exponential decay + floor + noise"), with one extra term, `overfit = np.where(epochs > 25, 0.006 * (epochs - 25), 0.0)` — it grows linearly from epoch 25 on, simulating the overfitting moment when "the model starts memorizing the training set and turns back upward on the validation set".

**The key point**: all of this code generates **made-up toy data**, not the log of any real training run; the goal is to let you see clearly, in one picture, "what a typical overfitting curve looks like". When you train the real `KeywordCNN` in Aurora, `train_loss`/`val_loss` will be lists of numbers actually recorded by the training loop; just `plt.plot()` them — there's no need (and no use) for an exponential formula like this.


In [ ]:
# ── Demo: generate typical train/val loss curves (including an overfitting phase) ──
np.random.seed(42)
epochs = np.arange(1, 61)

train_loss = 1.0 * np.exp(-0.08 * epochs) + 0.05 + 0.01 * np.random.randn(60)
val_decay  = 1.0 * np.exp(-0.05 * epochs) + 0.15
overfit    = np.where(epochs > 25, 0.006 * (epochs - 25), 0.0)
val_loss   = val_decay + overfit + 0.015 * np.random.randn(60)

best_epoch = int(np.argmin(val_loss)) + 1

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(epochs, train_loss, label='train loss', color='#2980b9', lw=2)
ax.plot(epochs, val_loss,   label='val loss',   color='#e74c3c', lw=2)
ax.axvline(best_epoch, color='#27ae60', ls='--', lw=1.5,
           label=f'early stop @ epoch {best_epoch}')
ax.fill_between(epochs,
    np.minimum(train_loss, val_loss),
    np.maximum(train_loss, val_loss),
    alpha=0.12, color='#8e44ad', label='generalization gap')
ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('Loss', fontsize=11)
ax.set_title('Train vs Val Loss — early signs of overfitting', fontsize=13)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

gap_at_best = val_loss[best_epoch-1] - train_loss[best_epoch-1]
print(f'✅ best epoch = {best_epoch}')
print(f'   val_loss_min  = {val_loss[best_epoch-1]:.4f}')
print(f'   generalization gap = {gap_at_best:.4f}')
print('   → the bigger the purple region, the worse the overfitting; early stopping saves the best weights at the green line')

In [ ]:
# Visualize gradient norms
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# Demo: define a small model and run one forward + backward pass to fill in the gradients
model = nn.Sequential(
    nn.Linear(40, 64), nn.ReLU(),
    nn.Linear(64, 32), nn.ReLU(),
    nn.Linear(32, 10)
)
x_demo = torch.randn(16, 40)
y_demo = torch.randint(0, 10, (16,))
loss = nn.CrossEntropyLoss()(model(x_demo), y_demo)
loss.backward()

grad_norms = []
for name, param in model.named_parameters():
    if param.grad is not None:
        grad_norms.append((name, param.grad.norm().item()))

names = [n for n, _ in grad_norms]  # use the full parameter path to tell the layers apart
norms = [v for _, v in grad_norms]

plt.figure(figsize=(10, 4))
plt.bar(range(len(norms)), norms)
plt.xticks(range(len(names)), names, rotation=45, ha='right')
plt.ylabel('gradient L2 norm')
plt.title('Gradient norm of each parameter')
plt.tight_layout()
plt.show()

### Diagnostic criteria for gradient norms — telling exploding / vanishing / healthy apart

A gradient's L2 norm is **the square root of the sum of squares of all that parameter's gradient elements**:

$$\|\nabla\theta\|_2 = \sqrt{\sum_i (\partial L / \partial \theta_i)^2}$$

This single number directly reflects the "strength of the gradient signal" that layer of parameters receives. Different values mean different things:

| Gradient norm range | Diagnosis | Symptoms and consequences | What to adjust |
|-----------|------|---------|--------|
| **< 0.0001** | 🔴 vanishing gradients | the gradient is too small; the parameters barely update | • check the activation functions (ReLU/Sigmoid/Tanh) <br> • use residual connections or batch norm |
| **0.0001 ~ 0.01** | 🟡 weak gradients | slow learning, long time to converge | • increase the learning rate <br> • check the data preprocessing (normalization) |
| **0.01 ~ 1.0** | 🟢 healthy gradients | the typical range for normal training | keep the current settings |
| **1.0 ~ 10** | 🟡 strong gradients | learning may be fast but unstable | • reduce the learning rate <br> • add gradient clipping <br> • add regularization (dropout/L2) |
| **> 10** | 🔴 exploding gradients | parameter updates are too large; the loss oscillates or diverges | • reduce the learning rate right away <br> • turn on gradient clipping: `torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)` <br> • check whether the activation functions are saturating |

#### How to judge in practice

Usually you watch the distribution of gradient norms early in training (the first 5 epochs):

1. **Normal**: every layer's gradient norm is between 0.01 and 1.0, steady or changing slowly
2. **Signs of vanishing gradients**: the gradient norms of the early layers (the first convolution layers, far from the output) creep toward 0, while those of the layers near the output stay > 0.1
3. **Signs of exploding gradients**: some layer's gradient norm suddenly jumps to 100+, and the loss turns into NaN or Inf at the same time

#### Code example: monitoring gradient norms

```python
for batch_idx, (x, y) in enumerate(train_loader):
    logits = model(x)
    loss = criterion(logits, y)
    loss.backward()

    # check the gradient norms
    for name, param in model.named_parameters():
        if param.grad is not None:
            norm = param.grad.norm().item()
            if norm > 10:
                print(f"⚠️  {name} exploding gradient: {norm:.2f}")
            elif norm < 1e-4:
                print(f"⚠️  {name} vanishing gradient: {norm:.2e}")

    optimizer.step()
    optimizer.zero_grad()
```



In [ ]:
# Visualize the weight distributions (histograms)
fig, axes = plt.subplots(2, 3, figsize=(12, 6))
axes = axes.flatten()

params_to_plot = [(name, param) for name, param in model.named_parameters()
                  if param.requires_grad and param.dim() >= 2][:6]

for i, (name, param) in enumerate(params_to_plot):
    axes[i].hist(param.detach().cpu().numpy().flatten(), bins=50, edgecolor='none')
    axes[i].set_title(name.split('.')[-1])
    axes[i].set_xlabel('weight value')
    axes[i].set_ylabel('count')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Weight distribution histograms')
plt.tight_layout()
plt.show()

## ✏️ Exercise: implement early_stopping()

Given a list of validation losses and a patience parameter, find **the epoch that early stopping returns** (counting from 1).

The rule: when the val_loss of the last `patience` consecutive epochs has been **higher** than the current minimum, return the epoch number of the current minimum (the best epoch).

Hint: use one variable to track the minimum seen so far and another as a counter.


### Early-stopping logic in detail: a state-transition table

The key to understanding early stopping is tracking how **two variables** change:
1. `best_loss`: the smallest val_loss seen so far
2. `no_improve`: the count of consecutive epochs without improvement

**The rules**:
- when `loss < best_loss`, a new best has been found → update `best_loss` and `best_epoch`, and **reset** `no_improve=0`
- when `loss >= best_loss`, there's no improvement → **increment** `no_improve += 1`
- when `no_improve >= patience`, early stopping fires → **return** `best_epoch`

**Example 1: losses_a = [0.9, 0.7, 0.5, 0.6, 0.65, 0.7, 0.72, 0.75], patience=3**

| epoch | val_loss | best_loss | Improved? | no_improve | Action | Return value |
|-------|----------|-----------|---------|-----------|------|--------|
| 1 | 0.90 | ∞ | ✓ new minimum | 0 | update best_epoch=1 | — |
| 2 | 0.70 | 0.90 | ✓ new minimum | 0 | update best_epoch=2 | — |
| 3 | 0.50 | 0.70 | ✓ new minimum | 0 | update best_epoch=3 | — |
| 4 | 0.60 | 0.50 | ✗ no improvement | 1 | continue | — |
| 5 | 0.65 | 0.50 | ✗ no improvement | 2 | continue | — |
| 6 | 0.70 | 0.50 | ✗ no improvement | 3 | no_improve=3 ≥ patience | **return 3** |

→ **Early stopping fires at epoch 6 and returns epoch 3 (the best epoch)**

**Example 2: losses_b = [0.9, 0.8, 0.7, 0.6, 0.5, 0.4], patience=2**

| epoch | val_loss | best_loss | Improved? | no_improve | Action |
|-------|----------|-----------|---------|-----------|------|
| 1 | 0.90 | ∞ | ✓ new minimum | 0 | update best_epoch=1 |
| 2 | 0.80 | 0.90 | ✓ new minimum | 0 | update best_epoch=2 |
| 3 | 0.70 | 0.80 | ✓ new minimum | 0 | update best_epoch=3 |
| 4 | 0.60 | 0.70 | ✓ new minimum | 0 | update best_epoch=4 |
| 5 | 0.50 | 0.60 | ✓ new minimum | 0 | update best_epoch=5 |
| 6 | 0.40 | 0.50 | ✓ new minimum | 0 | update best_epoch=6 |

→ **The sequence keeps falling, never fires, and returns 6 (the last best value)**

**Code skeleton**:
```python
best_epoch = 1
best_loss  = float('inf')
no_improve = 0
for epoch, loss in enumerate(val_losses, start=1):
    if loss < best_loss:
        best_loss = loss
        best_epoch = epoch
        no_improve = 0          # ← key: reset when a new minimum is found
    else:
        no_improve += 1         # ← key: increment when there's no improvement
        if no_improve >= patience:
            return best_epoch   # ← key: the trigger condition
return best_epoch               # ← fallback: the sequence ended without firing
```



In [ ]:
def early_stopping(val_losses: list, patience: int) -> int:
    """
    Return the best epoch (1-based) at the point early stopping fires.
    If it never fires over the whole sequence, return the epoch with the smallest val_loss.

    Parameters
    ----------
    val_losses : list[float]  validation loss for each epoch
    patience   : int          how many epochs in a row without improvement before stopping

    Returns
    -------
    int  the best epoch number (1-based)
    """
    # ── TODO: complete the implementation ──
    best_epoch = 1
    best_loss  = float('inf')
    no_improve = 0
    for epoch, loss in enumerate(val_losses, start=1):
        raise NotImplementedError("Delete this line and complete the implementation")
    return best_epoch


# ── Checks ──
try:
    losses_a = [0.9, 0.7, 0.5, 0.6, 0.65, 0.7, 0.72, 0.75]  # rises after epoch 3, patience=3 → stops at epoch 3
    result_a = early_stopping(losses_a, patience=3)
    assert result_a == 3, f'expected 3, got {result_a}'

    losses_b = [0.9, 0.8, 0.7, 0.6, 0.5, 0.4]  # keeps falling, returns the last epoch
    result_b = early_stopping(losses_b, patience=2)
    assert result_b == 6, f'expected 6, got {result_b}'

    losses_c = [1.0, 0.5, 0.6, 0.7, 0.8, 0.9]  # patience=2, epoch 2 is best
    result_c = early_stopping(losses_c, patience=2)
    assert result_c == 2, f'expected 2, got {result_c}'

    print('✅ All asserts pass! early_stopping() is implemented correctly.')
    print(f'   losses_a(patience=3) → best_epoch = {result_a}')
    print(f'   losses_b(patience=2) → best_epoch = {result_b}')
    print(f'   losses_c(patience=2) → best_epoch = {result_c}')
except (NotImplementedError, TypeError):
    print('⚠️  Complete the TODO in early_stopping() first, then run the checks.')
except AssertionError as e:
    print(f'❌ Check failed: {e}')
    print('   Hint: check the condition for updating best_epoch, and when the no_improve counter is reset.')


## Bridge: comparing mel spectrograms of misclassified samples

The three kinds of plots above — the DAG, the loss curves, and the gradient norms / weight distributions — have shown the debugging toolkit for the training process.
Next, the **confusion matrix** and a **mel spectrogram comparison** go one step further: when the model mistakes "no" for "yes", which specific features of the spectrum trip the classifier up?


### Why do we need the mel frequency transform? Human hearing isn't linear in frequency

**An intuitive analogy**: imagine two tuning forks.
- going from 100 Hz → 200 Hz feels like a big jump in pitch
- going from 8000 Hz → 8100 Hz, we can hardly hear any difference

Why? Because **the ear is more sensitive at low frequencies, and its ability to tell high frequencies apart is weaker**. If you analyze speech on a linear frequency axis, you waste a lot of computation on fine high-frequency differences, while the important low-frequency information gets "squashed".

**The mel scale**: transform frequency onto a "mel" scale that matches human perception, so that equal mel steps sound equally far apart.

$$\text{mel}(f) = 2595 \log_{10}\left(1 + \frac{f}{700}\right)$$

This is an empirical formula (rooted in hearing experiments by Stevens and colleagues in 1937). What the key numbers mean:
- **700**: a reference frequency (in Hz)
- **2595**: a scaling constant, chosen so that 1000 Hz maps to 1000 mel
- **$\log_{10}$**: the logarithm "squashes" the high frequencies

**Examples**:
- 80 Hz  → mel ≈ $2595 \log_{10}(1 + 80/700) \approx 122$
- 1000 Hz → mel ≈ $2595 \log_{10}(1 + 1000/700) \approx 1000$
- 8000 Hz → mel ≈ $2595 \log_{10}(1 + 8000/700) \approx 2840$

Notice: 80→1000 Hz spans about 878 mel, while 1000→8000 Hz spans only about 1840 mel, even though that frequency gap is more than seven times larger.

#### Designing the triangular filters

On the mel scale, sample with **equally spaced triangular filters**. Each triangle peaks at its center frequency (response = 1.0) and falls linearly to 0 on both sides. Why?

1. **Computational efficiency**: applying a triangle to the spectrum is just a weighted sum (no expensive Fourier transform involved)
2. **Human hearing**: the ear's response to frequency is roughly "local", and a triangle's width roughly matches a critical band
3. **Feature extraction**: overlapping neighboring triangles guarantee a smooth frequency response, so no information falls through the cracks

**The code logic**:
- spread `n_mels` triangle centers evenly along the mel axis
- each triangle's base width is set by its neighbors (they overlap by ~50%)
- interpolate linearly over the spectrum to get each triangle's response at every frequency point



## 3. The confusion matrix — computing it by hand and visualizing it

Confusion matrix `CM[i, j]` = the number of samples whose true class is `i` and which are predicted as `j`.
- **diagonal**: correct classifications; **off-diagonal**: misclassifications
- **Recall** = `CM[i,i] / sum(CM[i,:])` — the fraction of this class that is correctly recognized
- **Precision** = `CM[i,i] / sum(CM[:,i])` — of the predictions of this class, the fraction that really are this class

Below we accumulate it by hand with `np.zeros`, **without sklearn**.


In [ ]:
# ── A small example: what axis=1 and keepdims mean ──
# A common confusion-matrix operation: CM.sum(axis=1, keepdims=True) — sum each row

# an example with small numbers
CM_tiny = np.array([
    [15,  0,  1],  # row 0: where true class 0 went (15 right, 0 sent to 1, 1 sent to 2)
    [ 2, 18,  0],  # row 1: where true class 1 went (2 sent to 0, 18 right, 0 sent to 2)
    [ 0,  1, 19],  # row 2: where true class 2 went (0 sent to 0, 1 sent to 1, 19 right)
])

print('The matrix CM_tiny (3 rows, 3 columns):')
print(CM_tiny)
print(f'shape: {CM_tiny.shape}\n')

# axis=1 means add along the column direction (i.e. sum each row)
row_sums = CM_tiny.sum(axis=1)
print('CM.sum(axis=1) — add along the columns (= sum each row):')
print(f'{row_sums}  shape={row_sums.shape}')
print('  → [15+0+1=16, 2+18+0=20, 0+1+19=20]\n')

# keepdims=True keeps the axis (both rows and columns remain; the summed axis just becomes size 1)
row_sums_2d = CM_tiny.sum(axis=1, keepdims=True)
print('CM.sum(axis=1, keepdims=True) — the result stays 2D (3 rows, 1 column):')
print(row_sums_2d)
print(f'shape: {row_sums_2d.shape}\n')

# row normalization: divide each row by its sum
CM_norm = CM_tiny.astype(float) / row_sums_2d
print('Row-normalized (divided by the row sums):')
print(CM_norm)
print('every row sums to 1.0 (each row is a probability distribution for that class)\n')

print('✅ Key concepts:')
print('  · axis=1 = operate along the columns = aggregate the elements of each row')
print('  · axis=0 = operate along the rows = aggregate the elements of each column')
print('  · keepdims=True keeps the number of dimensions (2D→2D), handy for broadcasting')


In [ ]:
# ── Confusion matrix: computed by hand (numpy only) + imshow visualization ──
# Simulate a 10-class keyword classifier's predictions on 200 test samples
np.random.seed(7)
CLASSES = ['yes','no','up','down','left','right','on','off','stop','go']
N_CLASSES = len(CLASSES)
N_SAMPLES = 200

y_true = np.random.randint(0, N_CLASSES, N_SAMPLES)
# give the classifier a systematic confusion: 'no'(1) is mistaken for 'yes'(0) 30% of the time
y_pred = y_true.copy()
for idx in range(N_SAMPLES):
    if y_true[idx] == 1 and np.random.rand() < 0.30:
        y_pred[idx] = 0  # no → yes misclassification
    elif y_true[idx] != 1 and np.random.rand() < 0.10:
        y_pred[idx] = np.random.randint(0, N_CLASSES)

# accumulate the confusion matrix by hand
CM = np.zeros((N_CLASSES, N_CLASSES), dtype=int)
for t, p in zip(y_true, y_pred):
    CM[t, p] += 1

# Row normalization: divide each row by its sum → each cell becomes "of the samples whose true class is i,
# what fraction were predicted as j" — the recall view; each row sums to 1
CM_norm_row = CM.astype(float) / (CM.sum(axis=1, keepdims=True) + 1e-9)

# Column normalization: divide each column by its sum → each cell becomes "of the samples the model predicted as j,
# what fraction truly belong to class i" — the precision view; each column sums to 1
CM_norm_col = CM.astype(float) / (CM.sum(axis=0, keepdims=True) + 1e-9)

fig, axes = plt.subplots(1, 3, figsize=(19, 5))

# left: the count matrix
im0 = axes[0].imshow(CM, cmap='Blues')
axes[0].set_title('Confusion matrix (counts)', fontsize=12)
axes[0].set_xticks(range(N_CLASSES)); axes[0].set_xticklabels(CLASSES, rotation=45, ha='right')
axes[0].set_yticks(range(N_CLASSES)); axes[0].set_yticklabels(CLASSES)
axes[0].set_xlabel('Predicted class'); axes[0].set_ylabel('True class')
for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        axes[0].text(j, i, CM[i,j], ha='center', va='center', fontsize=8,
                     color='white' if CM[i,j] > CM.max()*0.6 else 'black')
plt.colorbar(im0, ax=axes[0])

# middle: the row-normalized matrix (recall view) — read along a row: where did this true class's samples go
im1 = axes[1].imshow(CM_norm_row, cmap='Reds', vmin=0, vmax=1)
axes[1].set_title('Row-normalized = Recall\n"of the samples that are truly i, how many were recognized"', fontsize=11)
axes[1].set_xticks(range(N_CLASSES)); axes[1].set_xticklabels(CLASSES, rotation=45, ha='right')
axes[1].set_yticks(range(N_CLASSES)); axes[1].set_yticklabels(CLASSES)
axes[1].set_xlabel('Predicted class'); axes[1].set_ylabel('True class')
for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        axes[1].text(j, i, f'{CM_norm_row[i,j]:.2f}', ha='center', va='center', fontsize=7,
                     color='white' if CM_norm_row[i,j] > 0.6 else 'black')
plt.colorbar(im1, ax=axes[1])

# right: the column-normalized matrix (precision view) — read down a column: when the model says j, how often is it right
im2 = axes[2].imshow(CM_norm_col, cmap='Greens', vmin=0, vmax=1)
axes[2].set_title('Column-normalized = Precision\n"of the samples predicted as j, how many really are j"', fontsize=11)
axes[2].set_xticks(range(N_CLASSES)); axes[2].set_xticklabels(CLASSES, rotation=45, ha='right')
axes[2].set_yticks(range(N_CLASSES)); axes[2].set_yticklabels(CLASSES)
axes[2].set_xlabel('Predicted class'); axes[2].set_ylabel('True class')
for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        axes[2].text(j, i, f'{CM_norm_col[i,j]:.2f}', ha='center', va='center', fontsize=7,
                     color='white' if CM_norm_col[i,j] > 0.6 else 'black')
plt.colorbar(im2, ax=axes[2])

plt.suptitle('KeywordCNN confusion matrix — simulated data (no→yes misclassification rate = 30%)', fontsize=13)
plt.tight_layout()
plt.show()

# print recall and precision
recall    = np.diag(CM) / (CM.sum(axis=1) + 1e-9)
precision = np.diag(CM) / (CM.sum(axis=0) + 1e-9)
print('📊 Recall / Precision per class:')
for c, r, p in zip(CLASSES, recall, precision):
    print(f'   {c:6s}  recall={r:.2f}  precision={p:.2f}')
print()
print('✅ CM["no","yes"] is especially high → "no" is often mistaken for "yes"')
print('   → together with the mel spectrogram comparison below, this points to the root cause: with too little energy in the nasal region, the spectrum looks like "yes"')
print()
print('📖 When do you read rows (recall), and when columns (precision)?')
print('   · if you care "of the samples where this class should appear, how many were found" → read the row (Recall)')
print('     e.g. if a voice assistant fails to recognize the command "stop" (a miss), the cost is high, so watch')
print('     the recall of the "stop" row — it measures "of the samples where someone really said stop, how many the model caught"')
print('   · if you care "once the model makes this prediction, how far can you trust it" → read the column (Precision)')
print('     e.g. in a smart home, once the model predicts "go" it really carries out the action; if that prediction is often')
print('     wrong (other words mistaken for go), watch the precision of the "go" column')
print('   · back to "no mistaken for yes": the two views describe two different things:')
print('     - the "no" row → no has low recall (30% of the no samples that should have been recognized were missed)')
print('     - the "yes" column → yes has its precision dragged down (many mislabeled no samples landed in the')
print('       yes pile, so when the model says "it\'s yes", that claim is less trustworthy)')


In [ ]:
# ── Parameter experiment: synthesize keyword mel images and compare a misclassification ──
# No real audio needed; sums of sines mimic the rough spectral features of yes/no

SR = 16000
N_MELS = 40      # ✏️ try 20 / 40 / 80 and watch how the frequency resolution changes
HOP    = 160     # ✏️ try 80 / 160 / 320 and watch how the time resolution changes
N      = int(SR * 0.5)  # 0.5 seconds

def synth_keyword(freqs, amps, n=N, sr=SR):
    t = np.linspace(0, n / sr, n)
    sig = sum(a * np.sin(2 * np.pi * f * t) for f, a in zip(freqs, amps))
    return sig / (np.abs(sig).max() + 1e-9)

def naive_mel(sig, sr=SR, n_mels=N_MELS, hop=HOP, n_fft=512):
    frames = [sig[i:i+n_fft] * np.hanning(n_fft)
              for i in range(0, len(sig) - n_fft, hop)]
    specs  = np.array([np.abs(np.fft.rfft(f))**2 for f in frames]).T
    freq_bins = np.linspace(0, sr/2, n_fft//2 + 1)
    mel_low  = 2595 * np.log10(1 + 80   / 700)
    mel_high = 2595 * np.log10(1 + 8000 / 700)
    mel_pts  = np.linspace(mel_low, mel_high, n_mels + 2)
    hz_pts   = 700 * (10 ** (mel_pts / 2595) - 1)
    fb = np.zeros((n_mels, n_fft//2 + 1))
    for m in range(1, n_mels + 1):
        f0, f1, f2 = hz_pts[m-1], hz_pts[m], hz_pts[m+1]
        for k, f in enumerate(freq_bins):
            if f0 <= f < f1:  fb[m-1, k] = (f - f0) / (f1 - f0 + 1e-12)
            elif f1 <= f <= f2: fb[m-1, k] = (f2 - f) / (f2 - f1 + 1e-12)
    return np.log(fb @ specs + 1e-9)

# yes: high-frequency consonant [3500, 4200] + mid-frequency vowel [500, 900]
yes_sig    = synth_keyword([3500, 4200, 500,  900], [0.6, 0.4, 0.8, 0.5])
# no: low-frequency nasal [200, 400] + mid-frequency vowel [600, 1000]
no_sig     = synth_keyword([200,  400,  600, 1000], [0.7, 0.5, 0.8, 0.4])
# a misclassified no: very weak nasal, strong high-frequency noise, so the model mistakes it for yes
no_bad_sig = synth_keyword([200,  4000, 500,  900], [0.1, 0.5, 0.7, 0.5])

mel_yes    = naive_mel(yes_sig)
mel_no     = naive_mel(no_sig)
mel_no_bad = naive_mel(no_bad_sig)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
items = [
    (mel_yes,    'yes, correct sample',         '#2980b9'),
    (mel_no,     'no, correct sample',           '#27ae60'),
    (mel_no_bad, 'no mistaken for yes\n(consonant region looks like yes)', '#e74c3c'),
]
for ax, (mel, title, bc) in zip(axes, items):
    im = ax.imshow(mel, aspect='auto', origin='lower',
                   cmap='magma', interpolation='nearest')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Time frame', fontsize=9)
    ax.set_ylabel('Mel filter index', fontsize=9)
    for spine in ax.spines.values():
        spine.set_edgecolor(bc); spine.set_linewidth(2.5)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.suptitle(
    f'Mel image comparison  (n_mels={N_MELS}, hop={HOP})  — red frame = misclassified sample',
    fontsize=12, y=1.02
)
plt.tight_layout()
plt.show()

print('✅ What to look for:')
print('  · correct yes (blue): a clear band of energy in the high-frequency filters (top)')
print('  · correct no (green): nasal energy in the low-frequency filters (bottom)')
print('  · misclassified no (red): the low-frequency nasal disappears and a high-frequency band appears → very similar to yes')
print(f'  · change N_MELS from {N_MELS} to 80 to see how higher frequency resolution affects how well they separate')

### 🎛 How should you actually choose N_MELS and HOP? — from "just try some" to choosing on principle

The comments in the code above only say "✏️ try N_MELS=20/40/80" and "✏️ try HOP=80/160/320 and watch what changes", but they don't tell you **which way to turn the knob, or why this lesson fixes them at 40 and 160**. Here is a set of practical rules of thumb.

#### N_MELS: how many slices along the frequency axis

Imagine describing a continuously changing rainbow with a grid of mosaic tiles. **The more tiles (N_MELS)**, the finer the differences you can tell between neighboring colors (higher frequency resolution); but once the tiles outnumber the information actually in the picture, what "each tile can see" drifts toward random noise — the model may learn not the real pattern but noise patterns that happened to appear in the training set, which is why the risk of **overfitting** grows. Conversely, with too few tiles, two genuinely different colors get forced into one, and the model simply can't tell them apart (underfitting, information lost).

So the principle is simple: **use as many tiles as your data and model capacity can support**.

| N_MELS | When to use it | Example |
|---|---|---|
| about 20 | little data, a simple task (e.g. wake-word detection with a few keywords), embedded devices with limited compute | smart-speaker wake words |
| 40 | the **balance point** for medium data and medium-complexity tasks; the most common default in industry | a common setting in speech toolkits such as Kaldi; this lesson's KeywordCNN |
| 80 or more | large-scale data, full speech-recognition systems chasing high accuracy | OpenAI Whisper uses 80 mel bands |

That's why this lesson fixes `N_MELS = 40`: this is a keyword-spotting task with neither very little nor very much data, and 40 is a compromise that's hard to get wrong — you can of course change it to 80 and try, but you'll usually find the validation accuracy gains little, and may even drop slightly because of overfitting.

#### HOP: how often to look at a frame along the time axis

Another analogy: taking a burst of photos to record someone running. **If the gap between shots (the hop) is too long**, you miss key in-between movements (poor time resolution; a fast consonant might be "skipped" entirely); **if it's too short**, you get piles of almost identical photos, wasting storage and compute for very little new information.

Rule of thumb: **HOP is usually 1/4 of N_FFT**. This lesson has `N_FFT=512`, so in theory `HOP≈128`; the code actually uses `HOP=160`, because at `SR=16000` (a 16 kHz sample rate) 160 samples is exactly **10 milliseconds** — a very common "round-number time" convention in speech processing (toolkits such as Kaldi default to a 10 ms frame shift), which fits the phonetic intuition about phoneme durations better than insisting on "exactly 1/4 of N_FFT" (the steady parts of most consonants and vowels last tens of milliseconds, and a 10 ms step is enough to capture how they change).

| Hyperparameter | Cost of too small | Cost of too large | This lesson's choice |
|---|---|---|---|
| N_MELS | not enough frequency resolution; similar phonemes can't be told apart (underfitting) | higher risk of overfitting, slower training | 40 (the balance point for medium data) |
| HOP | compute explodes; neighboring frames are highly redundant | worse time resolution; fast phonemes get "smeared out" | 160 (10 ms at 16 kHz, the speech-processing convention) |

In one sentence: **both hyperparameters follow "how much data, how hard the task, how much overfitting you can afford"; there is no fixed value that's right everywhere — 40 and 160 are just a sensible starting point for this lesson's task, and the best way to learn is to change them and watch how the validation accuracy moves.**


### Mel spectrogram comparison: the frequency-domain features of a misclassified sample

Below is a **synthetic demonstration**: sums of sine waves mimic the rough spectral features of "yes" and "no", which are then passed through the mel filters.

⚠️ **Important**: the frequencies below are synthetic, designed to show **a possible root cause of a classification error**; they **do not represent real English pronunciation**. Real "yes" and "no" contain complex time-varying spectra, formants and intricate consonants; this demo is a highly simplified sketch.

**Demo design**:
- **"yes", correct sample**: contains a **high-frequency consonant** [3500, 4200 Hz] and a **mid-frequency vowel** [500, 900 Hz]
  - the high-frequency consonant is characteristic of the English /s/ sound
- **"no", correct sample**: contains a **low-frequency nasal** [200, 400 Hz] and a **mid-frequency vowel** [600, 1000 Hz]
  - the low-frequency nasal is characteristic of the /n/ sound
- **"no", misclassified sample** (mistaken for "yes"): a very weak nasal, **replaced by high-frequency noise** [around 4000 Hz]
  - this mimics a noisy environment where the nasal is drowned out and high-frequency energy suddenly appears
  - as a result its mel image looks like "yes" → the classifier is confused

#### Digging deeper: why exactly [3500, 4200] and [200, 400]? Acoustic fact, or picked at random?

The answer: **neither an exact reproduction of real speech nor a number pulled out of thin air — they were deliberately chosen to be "extreme", to exaggerate the difference so you can see it at a glance.** The design logic goes like this:

1. **First set the general direction**: in real speech, fricatives (such as /s/) really do concentrate more energy in the mid-to-high frequencies, and nasals (such as /n/) really do concentrate more energy in the low frequencies — this is the **rough starting point** for choosing these two bands, not something made up.
2. **Then deliberately push them apart**: in real speech, the spectra of different phonemes actually overlap, transition and vary from person to person a great deal (accent, speaking rate and sex all make the spectrum drift). If this demo copied real spectra, "yes" and "no" might look very close in the picture, which isn't good for teaching. So here **the two bands are deliberately chosen far apart** (3500–4200 Hz is mid-to-high frequency, 200–400 Hz is low frequency, with about three octaves in between) — the goal is simply to make the difference between the two synthetic signals in the mel image **big enough to spot with the naked eye**, so you can "read" from the picture where the model is likely to go wrong.
3. **In other words**: if the two bands were close together (say, both mid-frequency), the difference in the mel images would be hard to see, and you couldn't see "why the misclassified sample looks like yes" — that is the real reason for these particular values; **it isn't a reproduction of acoustic measurements**.

The spectral structure of real speech is much more complex: each phoneme is made of several **formants** (usually written F1, F2, F3, ...), and the exact formant frequencies shift with the speaker (sex, accent), the speaking rate and the neighboring phonemes, rather than sitting in a fixed range. If you want to dig into the real spectral structure of vowels and consonants, look up the chapters on "formants" and "acoustic phonetics" in a phonetics textbook — they give frequency distributions measured from large collections of real recordings, rather than two ranges exaggerated for teaching effect as in this lesson.



In [ ]:
# ── A small example: how mel filters are built (a demo with 2–3 filters) ────────────────────
# Goal: get an intuitive feel for what "triangular filters equally spaced on the mel scale" means

SR = 16000
N_FFT = 512
freq_bins = np.linspace(0, SR/2, N_FFT//2 + 1)  # frequency axis: 0 ~ 8000 Hz

# Mel transform
mel_low  = 2595 * np.log10(1 + 80   / 700)    # lower bound: 80 Hz
mel_high = 2595 * np.log10(1 + 8000 / 700)    # upper bound: 8000 Hz
print(f'Frequency range: 80 ~ 8000 Hz')
print(f'Mel range: {mel_low:.1f} ~ {mel_high:.1f} mel\n')

# a simple demo: just 3 mel filters
N_MELS_DEMO = 3
mel_pts = np.linspace(mel_low, mel_high, N_MELS_DEMO + 2)
hz_pts  = 700 * (10 ** (mel_pts / 2595) - 1)

print(f'{N_MELS_DEMO} mel filters and their center frequencies (Hz):')
for m in range(1, N_MELS_DEMO + 1):
    f0, f1, f2 = hz_pts[m-1], hz_pts[m], hz_pts[m+1]
    print(f'  Filter {m}: left edge {f0:.0f} Hz → center {f1:.0f} Hz → right edge {f2:.0f} Hz')
print()

# build the triangular filters
fb = np.zeros((N_MELS_DEMO, len(freq_bins)))
for m in range(1, N_MELS_DEMO + 1):
    f0, f1, f2 = hz_pts[m-1], hz_pts[m], hz_pts[m+1]
    for k, f in enumerate(freq_bins):
        if f0 <= f < f1:  # rising edge
            fb[m-1, k] = (f - f0) / (f1 - f0 + 1e-12)
        elif f1 <= f <= f2:  # falling edge
            fb[m-1, k] = (f2 - f) / (f2 - f1 + 1e-12)

# plot the triangles
fig, ax = plt.subplots(figsize=(11, 4))
colors = ['#3498db', '#e74c3c', '#2ecc71']
for m in range(N_MELS_DEMO):
    ax.plot(freq_bins, fb[m, :], label=f'Filter {m+1}', color=colors[m], lw=2)
    # mark the center frequency
    center_hz = hz_pts[m+1]
    ax.axvline(center_hz, color=colors[m], ls='--', alpha=0.4)
    ax.text(center_hz, 1.05, f'{center_hz:.0f}', ha='center', fontsize=8)

ax.set_xlabel('Frequency (Hz)', fontsize=11)
ax.set_ylabel('Filter response', fontsize=11)
ax.set_title(f'{N_MELS_DEMO} triangular mel filters (equally spaced on the mel scale)', fontsize=12)
ax.set_xlim(0, 8000)
ax.set_ylim(0, 1.15)
ax.legend(loc='upper right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f'✅ Key observations:')
print(f'  • equally spaced on the mel scale → unevenly spaced on the frequency axis (dense at low frequencies, sparse at high)')
print(f'  • each filter is a triangle → it peaks at 1.0 at its center frequency')
print(f'  • neighboring filters overlap by ~50% → a smooth frequency response')
print(f'  • applied to a spectrum: `mel_output = fb @ freq_spectrum` extracts {N_MELS_DEMO} mel energy values')


## Wrap-up

This section used `naive_mel()` (hand-written triangular filters) to generate mel spectrograms, used a matplotlib DAG drawing to visualize the `data`/`grad` of `Value` nodes, and used synthetic signals to show the overfitting shape of train/val loss curves and when early stopping kicks in. The three pictures connect directly to Aurora's core modules: `aurora.audio.mel` (the mel filterbank) provides the real features, `Value.backward()` from `L54_value_autograd` fills in the real gradients, and the training loop from `L58_training_loop` produces the real loss curves. The next lesson (L66) moves on to the big picture of ASR: the three main architectures — CTC, attention-based encoder–decoder, and Whisper — and the overall design of Aurora's speech module.

In [ ]:
# ── Standalone mathematical assertions: checking the core properties of training visualization (no student implementation needed) ────────────────────────
import numpy as np

# 1. early_stopping without depending on your implementation: a hand-written reference version checks the logic
def _ref_early_stopping(val_losses, patience):
    best_loss, best_ep, no_imp = float('inf'), 1, 0
    for ep, loss in enumerate(val_losses, 1):
        if loss < best_loss:
            best_loss, best_ep, no_imp = loss, ep, 0
        else:
            no_imp += 1
            if no_imp >= patience:
                return best_ep
    return best_ep

cases = [
    ([0.9, 0.7, 0.5, 0.6, 0.65, 0.7, 0.72, 0.75], 3, 3),
    ([0.9, 0.8, 0.7, 0.6, 0.5, 0.4],               2, 6),
    ([1.0, 0.5, 0.6, 0.7, 0.8, 0.9],               2, 2),
]
for losses, pat, expected in cases:
    got = _ref_early_stopping(losses, pat)
    assert got == expected, f"losses={losses},pat={pat}: got={got},exp={expected}"
print(f"1 ✅  early_stopping reference implementation: all 3 test cases pass (patience=3→ep3, 2→ep6, 2→ep2)")

# 2. confusion-matrix diagonal = number of correct predictions; row normalization = recall
rng = np.random.default_rng(7)
n_cls = 4
y_true_q = rng.integers(0, n_cls, 100)
y_pred_q = y_true_q.copy()
# deliberately misclassify some
mask = rng.random(100) < 0.2
y_pred_q[mask] = rng.integers(0, n_cls, mask.sum())

CM_q = np.zeros((n_cls, n_cls), dtype=int)
for t, p in zip(y_true_q, y_pred_q): CM_q[t, p] += 1

acc_from_diag = CM_q.diagonal().sum() / CM_q.sum()
acc_direct = (y_true_q == y_pred_q).mean()
assert abs(acc_from_diag - acc_direct) < 1e-12
print(f"2 ✅  CM diagonal / total = accuracy = {acc_from_diag:.4f} (matches the direct computation)")

# 3. row normalization = recall per class
CM_norm_q = CM_q.astype(float) / (CM_q.sum(axis=1, keepdims=True) + 1e-9)
recall_q = CM_q.diagonal() / (CM_q.sum(axis=1) + 1e-9)
assert np.allclose(CM_norm_q.diagonal(), recall_q, atol=1e-10)
print(f"3 ✅  diagonal of the row-normalized CM = recall per class: {np.round(recall_q,3)}")

# 4. generalization gap = val_loss - train_loss > 0 (in the overfitting phase)
train_l = np.array([1.0, 0.7, 0.5, 0.4, 0.35, 0.3])
val_l   = np.array([1.0, 0.72, 0.55, 0.5, 0.6, 0.75])
gap = val_l - train_l
overfit_start = np.argmax(gap > 0.05) + 1   # the first epoch with gap > 0.05
assert overfit_start >= 1
print(f"4 ✅  generalization gap={np.round(gap,3)}, overfitting starts at epoch ≥ {overfit_start}")

# 5. gradient L2 norm = sqrt(sum(g²)): check the formula
import torch, torch.nn as nn
g_test = torch.tensor([3.0, 4.0])
norm_torch = g_test.norm().item()
norm_manual = float(np.sqrt((3**2 + 4**2)))
assert abs(norm_torch - norm_manual) < 1e-6
print(f"5 ✅  gradient L2 norm: ||[3,4]||₂ = √(9+16) = {norm_manual:.4f} (torch={norm_torch:.4f})")


In [ ]:
# ✏️ Self-review for this lesson
l65_review = {
    "dag_graph_reading":      None,  # can read the Value computation-graph DAG: node data/grad, gradients flowing back along the edges? True/False
    "loss_curve_overfit":     None,  # can spot where overfitting starts on train/val loss curves (val rises while train keeps falling)? True/False
    "early_stopping_impl":    None,  # early_stopping() correct, all 3 patience tests pass? True/False
    "grad_norm_diagnosis":    None,  # understand what gradient-norm plots diagnose: exploding/vanishing/healthy ranges? True/False
    "confusion_matrix_read":  None,  # can read a confusion matrix: diagonal = correct predictions, row-normalized = recall? True/False
}

unfilled = [k for k, v in l65_review.items() if v is None]
assert not unfilled, f'not filled in yet: {unfilled}'
weak = [k for k, v in l65_review.items() if v is False]
if weak:
    print(f'⚠️  needs more work: {weak}')
else:
    print('✅ L65 all clear! On to L66: ASR end to end')

---

→ **Next lesson** · [L66 · ASR end to end](../7_asr/L66_asr_overview.ipynb)

> Next lesson covers **ASR end to end**: acoustic model → language model → decoder; end-to-end vs classical pipelines.